# HW7P3 — Fairness audit

**Your name:**

Using a model from HW7P2, audit it for bias: report metrics by group, identify disparities, and propose a fairness-conscious redesign. Complete each `TODO` section.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

## Task 1: Model and groups

TODO: re-create your HW7P2 model and name the group column(s) you audit.

In [8]:
# TODO
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data = load_breast_cancer()
print(data.feature_names)
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

group_column = 'radius_group'
df[group_column] = pd.qcut(df['mean radius'], q=2, labels=['Group_A', 'Group_B'])

X = df.drop(columns=['target', group_column])
y = df['target']
groups = df[group_column]

X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, random_state=42
)

logreg = LogisticRegression(random_state=42)
logreg.fit(X_train, y_train)

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

print(groups_test.value_counts())

['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']
radius_group
Group_B    58
Group_A    56
Name: count, dtype: int64


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Task 2: Metrics by group

In [9]:
# TODO: accuracy, precision, recall, positive-prediction rate per group
from sklearn.metrics import accuracy_score, precision_score, recall_score

def evaluate_groups(model, X_test, y_test, groups_test):
    results = {}
    for group_name in groups_test.unique():

        mask = (groups_test == group_name)
        X_g = X_test[mask]
        y_g = y_test[mask]

        preds = model.predict(X_g)

        results[group_name] = {
            'Accuracy': accuracy_score(y_g, preds),
            'Precision': precision_score(y_g, preds, zero_division=0),
            'Recall': recall_score(y_g, preds, zero_division=0),
            'Positive-Prediction Rate': preds.mean()
        }
    return pd.DataFrame(results).T

print("=== Logistic Regression ===")
lr_group_metrics = evaluate_groups(logreg, X_test, y_test, groups_test)
display(lr_group_metrics)

print("\n=== Decision Tree ===")
dt_group_metrics = evaluate_groups(dt_model, X_test, y_test, groups_test)
display(dt_group_metrics)

=== Logistic Regression ===


,Accuracy,Precision,Recall,Positive-Prediction Rate
Group_A,1.000000,1.000000,1.0000,0.982143
Group_B,0.913793,0.789474,0.9375,0.327586



=== Decision Tree ===


,Accuracy,Precision,Recall,Positive-Prediction Rate
Group_A,0.964286,1.000000,0.963636,0.946429
Group_B,0.931034,0.833333,0.937500,0.310345


## Task 3: Disparities

TODO: the largest disparity, as a number.
Precision in logistic regression, 1.000000-0.789474=0.210526.

## Task 4: Redesign

TODO: describe the change, apply it, and show before and after.

In [12]:
# TODO
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

redesigned_model = make_pipeline(StandardScaler(), LogisticRegression(random_state=42, max_iter=10000))
redesigned_model.fit(X_train, y_train)

def evaluate_groups(model, X_test, y_test, groups_test):
    results = {}
    for group_name in groups_test.unique():
        mask = (groups_test == group_name)
        X_g = X_test[mask]
        y_g = y_test[mask]
        preds = model.predict(X_g)
        results[group_name] = {
            'Accuracy': accuracy_score(y_g, preds),
            'Precision': precision_score(y_g, preds, zero_division=0),
            'Recall': recall_score(y_g, preds, zero_division=0),
            'Positive-Prediction Rate': preds.mean()
        }
    return pd.DataFrame(results).T

print("=== Redesigned Logistic Regression ===")
new_group_metrics = evaluate_groups(redesigned_model, X_test, y_test, groups_test)
display(new_group_metrics)

=== Redesigned Logistic Regression ===


,Accuracy,Precision,Recall,Positive-Prediction Rate
Group_A,1.000000,1.000000,1.0000,0.982143
Group_B,0.948276,0.882353,0.9375,0.293103


## AI use disclosure

TODO: which AI tools (if any) you used and what you checked or changed. Write "None" if you did not use AI.
I used gemini. I checked the process of standard scaler and improvement of two models.